# BinSense — M3b · Track 2: Quality-Gated Auto-Label (the TREATMENT)

**Goal:** the same SAM + zero-shot boxes as Track 1, but a box (or a whole bin) must
survive **four gates** before it is trusted (every output box is tagged
`provenance=auto`):

- **Gate C — geometry sanity:** drop specks, bin-/frame-sized boxes, extreme-aspect slivers.
- **Gate B — cross-method agreement:** keep only boxes SAM *and* zero-shot both found (IoU ≥ thresh). Kills single-method hallucinations.
- **Gate A — count plausibility:** reject the whole bin if #boxes is implausible vs `EXPECTED_QUANTITY` (0 → fail; ≫ E → over-segmented).
- **Gate D — seed-model agreement (optional):** self-training; only if seed-detector boxes are supplied. Weak until the detector improves — off by default.

**Known risk we watch for (issue #2):** gates can bias the kept set toward *easy*
bins (Step 5 quantifies this). Logic lives in `tools/labeling/autolabel.py`; this
notebook orchestrates + visualizes. Eval-gold held out throughout.

In [ ]:
# Cell 1: Bootstrap — path resolution + data/code split
import sys, os, subprocess
from pathlib import Path

GITHUB_URL = 'https://github.com/rishib09/AmazonBinSense.git'
BRANCH     = 'm3b-auto-labeling'   # set 'master' after this milestone merges
DRIVE_ROOT = '/content/drive/MyDrive/Interview Kickstart/Capstone Project/Amazon BinSense'
LOCAL_DATA = r'G:\My Drive\Interview Kickstart\Capstone Project\Amazon BinSense\data'

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/AmazonBinSense')
    if (PROJECT_ROOT / '.git').exists():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH], check=False)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH], check=False)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', 'origin', BRANCH, '--ff-only'], check=False)
    else:
        subprocess.run(['git', 'clone', '--branch', BRANCH, GITHUB_URL, str(PROJECT_ROOT)], check=True)
    os.environ['BINSENSE_DATA_DIR'] = str(Path(DRIVE_ROOT) / 'data')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless', 'pandas', 'pyyaml', 'matplotlib'], check=True)
except ImportError:
    IN_COLAB = False
    if os.getenv('BINSENSE_DIR'):
        PROJECT_ROOT = Path(os.environ['BINSENSE_DIR'])
    else:
        _cwd = Path.cwd()
        PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
    if not os.getenv('BINSENSE_DATA_DIR') and Path(LOCAL_DATA).exists():
        os.environ['BINSENSE_DATA_DIR'] = LOCAL_DATA

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Running in:', 'Google Colab' if IN_COLAB else 'Local', '| ROOT:', PROJECT_ROOT)

In [ ]:
# Cell 2: Imports + paths — raw method dirs, track output dir, eval wall
import json
import pandas as pd
from pathlib import Path
from utils.env_utils import setup_env

cfg = setup_env(verbose=True)
IMAGES_DIR = cfg.images_dir
META_DIR   = cfg.metadata_dir
SEED_DIR   = cfg.labels_dir                       # 120-bin manual seed (ground truth)

# Raw per-method boxes (written by the GPU cells below; persist on Drive)
SAM_DIR = cfg.data_dir / 'labels_auto' / 'sam'
ZS_DIR  = cfg.data_dir / 'labels_auto' / 'zeroshot'
SAM_DIR.mkdir(parents=True, exist_ok=True)
ZS_DIR.mkdir(parents=True, exist_ok=True)

# The one wall we never cross: eval-gold bins are the M7 test set.
EVAL_IDS = set(pd.read_csv(cfg.splits_dir / 'eval.csv')['bin_id'].astype(str).str.zfill(5))
EXTEND   = pd.read_csv(cfg.splits_dir / 'extend.csv')['bin_id'].astype(str).str.zfill(5).tolist()
EXTEND   = [b for b in EXTEND if b not in EVAL_IDS]
print(f'extend bins to auto-label: {len(EXTEND)}   eval held out: {len(EVAL_IDS)}   seed labels: {len(list(SEED_DIR.glob("*.txt")))}')

## Step 1 — Reuse the raw method boxes from `03b`
No re-inference: `03b` already wrote SAM + zero-shot boxes to Drive. We just gate them differently.

In [ ]:
# Cell 3: Confirm raw method boxes exist
n_sam = len(list(SAM_DIR.glob('*.txt')))
n_zs  = len(list(ZS_DIR.glob('*.txt')))
print(f'SAM box files: {n_sam}   zero-shot box files: {n_zs}')
if not (n_sam and n_zs):
    print('Run notebook 03b Step 1 (Colab GPU) first — Track 2 gates SAM ∩ zero-shot.')

## Step 2 — Watch each gate act (no GPU)
Synthetic boxes make the gate behavior legible before running on real bins — the inline "test".

In [ ]:
# Cell 5: Engine sanity check (no GPU) — proves the geometry/agreement/NMS logic
from tools.labeling.autolabel import iou, nms, gate_geometry, cross_method_agreement, gate_count, GateParams

def _b(cx, cy, w, h, s=1.0):
    return {'cx': cx, 'cy': cy, 'w': w, 'h': h, 'score': s, 'source': 't'}

p = GateParams()
assert iou(_b(.5,.5,.2,.2), _b(.5,.5,.2,.2)) == 1.0
assert iou(_b(.5,.5,.2,.2), _b(.9,.9,.2,.2)) == 0.0
# geometry drops speck / full-frame / sliver, keeps a normal box
kept = gate_geometry([_b(.5,.5,.01,.01), _b(.5,.5,.99,.99), _b(.5,.5,.6,.01), _b(.5,.5,.1,.1)], p)
assert len(kept) == 1
# NMS collapses a near-duplicate
assert len(nms([_b(.5,.5,.2,.2,.9), _b(.51,.51,.2,.2,.5), _b(.1,.1,.1,.1,.8)], p.nms_iou)) == 2
# agreement keeps only the box both methods found
assert len(cross_method_agreement([_b(.3,.3,.2,.2), _b(.8,.8,.1,.1)], [_b(.31,.31,.2,.2)], p.agree_iou)) == 1
# count gate: 0 and gross over-seg rejected; visible<expected accepted
assert gate_count(0, 10, p)[0] is False and gate_count(30, 10, p)[0] is False and gate_count(8, 10, p)[0] is True
print('Engine sanity check PASSED — geometry, NMS, cross-method agreement, count gate all behave.')

## Step 3 — Run the gated pipeline
Each bin flows C → B → A. A bin that fails Gate A is dropped (or would route to human review), not injected as noise. The manifest records provenance for every bin.

In [ ]:
# Cell 6: Build Track 2 (gated) with provenance manifest
from tools.labeling.autolabel import build_track, GateParams

OUT2 = cfg.data_dir / 'labels_track2'
p = GateParams()          # tune agree_iou / count_over_frac against the gold set later
st2 = build_track('track2', SAM_DIR, ZS_DIR, OUT2, META_DIR, EXTEND, EVAL_IDS, p)
print('Track 2 (gated):', json.dumps(st2.as_dict(), indent=2))

man2 = pd.DataFrame(st2.manifest)
if len(man2):
    print(f"\naccepted bins: {st2.n_bins_out}   rejected: {st2.n_rejected}   boxes kept: {st2.n_boxes_out}")
    print('reject reasons:', st2.reject_reasons)
    display(man2.head(12))
else:
    print('No raw boxes yet — run 03b Step 1 on Colab first.')

## Step 4 — What each gate removed + provenance
Trace the funnel: raw (SAM+zero-shot) → geometry-kept → cross-method consensus → final. The drop at each stage is the gate doing its job.

In [ ]:
# Cell 7: Gate funnel + gated overlays
import matplotlib.pyplot as plt, cv2, numpy as np
from tools.labeling.overlay_check import draw_overlay, parse_label

if len(man2) and 'n_consensus' in man2:
    acc = man2[man2['passed']]
    funnel = {
        'raw (sam+zs)': int((man2['n_sam'] + man2['n_zs']).sum()),
        'geometry-kept': int((man2['n_sam_geom'] + man2['n_zs_geom']).sum()),
        'consensus (Gate B)': int(man2['n_consensus'].sum()),
        'final (passed Gate A)': int(man2['n_final'].sum()),
    }
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].bar(range(len(funnel)), list(funnel.values()), color='steelblue')
    ax[0].set_xticks(range(len(funnel))); ax[0].set_xticklabels(list(funnel), rotation=20, ha='right')
    ax[0].set_title('box funnel through the gates'); ax[0].set_ylabel('total boxes')
    for i, v in enumerate(funnel.values()):
        ax[0].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
    reasons = st2.reject_reasons
    if reasons:
        ax[1].bar(list(reasons), list(reasons.values()), color='indianred')
        ax[1].set_title('bin reject reasons (Gate A / geometry)'); ax[1].set_ylabel('#bins')
    plt.tight_layout(); plt.show()

    shown = list(acc['bin_id'])[:3]
    if shown:
        fig, axes = plt.subplots(1, len(shown), figsize=(5*len(shown), 5))
        for ax_, bid in zip(np.atleast_1d(axes), shown):
            out = cfg.base_dir / 'reports' / 'track2_overlays' / f'{bid}.jpg'
            draw_overlay(IMAGES_DIR/f'{bid}.jpg', parse_label(OUT2/f'{bid}.txt'), out)
            ax_.imshow(cv2.cvtColor(cv2.imread(str(out)), cv2.COLOR_BGR2RGB)); ax_.set_title(bid); ax_.axis('off')
        plt.suptitle('Track 2 gated boxes (cross-confirmed)'); plt.show()
else:
    print('Nothing to visualize yet.')

## Step 5 — Selection-bias check (the biggest risk)

Gates reject dense/occluded bins first, which can skew the kept set toward **easy**
bins and teach M4 an easy-only distribution. Compare `EXPECTED_QUANTITY` for
**accepted vs rejected** bins — if accepted bins are systematically lower-count, the
gates are cherry-picking and we must loosen them or hand-label hard bins.

In [ ]:
# Cell 8: accepted-vs-rejected EXPECTED_QUANTITY distribution
if len(man2) and man2['expected'].notna().any():
    acc = man2[(man2['passed']) & man2['expected'].notna()]['expected'].astype(float)
    rej = man2[(~man2['passed']) & man2['expected'].notna()]['expected'].astype(float)
    fig, ax = plt.subplots(figsize=(8, 4))
    bins = range(0, int(man2['expected'].max()) + 2)
    ax.hist([acc, rej], bins=bins, label=[f'accepted (n={len(acc)})', f'rejected (n={len(rej)})'],
            color=['seagreen', 'indianred'])
    ax.set_xlabel('EXPECTED_QUANTITY'); ax.set_ylabel('#bins'); ax.legend()
    ax.set_title('Selection bias: item-count of accepted vs rejected bins'); plt.show()
    if len(acc) and len(rej):
        print(f'median expected — accepted: {acc.median():.1f}   rejected: {rej.median():.1f}')
        skew = rej.median() - acc.median()
        print('⚠️  gates skew toward EASY (low-count) bins — loosen or hand-label hard bins.'
              if skew >= 2 else '✓ no strong easy-bin skew detected.')
else:
    print('Need gated results with EXPECTED_QUANTITY to assess selection bias.')

## Step 6 — Assemble the Track-2 training set (seed + gated auto, EXCLUDE eval)

In [ ]:
# Cell 9: Write data_track2.yaml (manual seed + track-2 gated labels)
import yaml, random

seed_ids = [p.stem for p in SEED_DIR.glob('*.txt') if p.stem not in EVAL_IDS]
auto_ids = [p.stem for p in OUT2.glob('*.txt')] if OUT2.exists() else []
labeled  = sorted(set(seed_ids) | set(auto_ids))
assert not (set(labeled) & EVAL_IDS), 'EVAL LEAKAGE in Track-2 training labels!'

if not labeled:
    print('No labels — run Step 3 (needs raw boxes from 03b) then re-run.')
else:
    random.Random(42).shuffle(labeled)
    n_val = max(1, int(0.15 * len(labeled)))
    val, train = labeled[:n_val], labeled[n_val:]
    YDIR = cfg.data_dir / 'yolo_track2'; YDIR.mkdir(parents=True, exist_ok=True)
    for name, ids in [('train.txt', train), ('val.txt', val)]:
        (YDIR / name).write_text('\n'.join(str(IMAGES_DIR / f'{b}.jpg') for b in ids))
    (YDIR / 'data.yaml').write_text(yaml.safe_dump({
        'path': str(cfg.data_dir), 'train': str(YDIR/'train.txt'), 'val': str(YDIR/'val.txt'),
        'nc': 1, 'names': {0: 'item'}}, sort_keys=False))
    print(f'seed={len(seed_ids)}  auto(track2 gated)={len(auto_ids)}  train={len(train)}  val={len(val)}')
    print('wrote', YDIR / 'data.yaml')

## A/B handoff to M4

Train M4 twice on the Track-0 gold set — once on `yolo_track1/data.yaml` (raw union),
once on `yolo_track2/data.yaml` (gated) — plus the original 120-seed baseline (mAP@50
0.255). Record **mAP@50** and **count-within-1** for each. The decision the video's
Level-3 deep-dive turns on: *does gating beat raw union, and by how much?* If not,
the honest finding is that consensus gating did not pay for itself on this data —
and that is a legitimate result, not a failure.

> **Label-merge note:** seed labels live in `data/labels/`, auto labels in
> `data/labels_track{1,2}/`. Before training, merge each track's auto labels + the
> seed into a single label root (or copy the seed into each track dir) so Ultralytics
> finds one `.txt` per image. Keep `provenance=auto` bins traceable via the manifest.